In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Identify the project directory
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() in ["notebooks", "data"]:
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

# Load the already-prepared debit dataset
DATA_DIR = PROJECT_DIR / "data"
debit_output = DATA_DIR / "debit_fraud_dataset.csv"

df = pd.read_csv(debit_output)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (5548196, 12)


Step 2 – Prepare features and target

In [2]:
# Extract transaction hour from the date
df["date"] = pd.to_datetime(df["date"])
df["transaction_hour"] = df["date"].dt.hour

# Select input features for fraud prediction
X = df[
    [
        "amount",
        "use_chip",
        "mcc",
        "transaction_hour"
    ]
]

# Select the target variable (True = Fraud, False = Genuine)
y = df["fraud_label"]

# Display dataset dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (5548196, 4)
Target shape: (5548196,)

Target distribution:
fraud_label
False    5540732
True        7464
Name: count, dtype: int64


Step 3 – Split data into training and testing sets
Purpose: Divide our dataset into two parts:
- Training data (80%): Used by the model to learn patterns.
- Testing data (20%): Used to evaluate the model on unseen transactions.

In [3]:
from sklearn.model_selection import train_test_split

# Split data: 80% for training and 20% for testing
# stratify=y maintains the fraud/genuine ratio in both datasets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the resulting dataset sizes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining fraud distribution:")
print(y_train.value_counts())

print("\nTesting fraud distribution:")
print(y_test.value_counts())

Training features: (4438556, 4)
Testing features: (1109640, 4)

Training fraud distribution:
fraud_label
False    4432585
True        5971
Name: count, dtype: int64

Testing fraud distribution:
fraud_label
False    1108147
True        1493
Name: count, dtype: int64


Step 4 – Encode categorical features

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features
numerical_features = ["amount", "mcc", "transaction_hour"]
categorical_features = ["use_chip"]

# Configure preprocessing for both feature types
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing configured successfully!")

Preprocessing configured successfully!


Step 5 – Train the Logistic Regression model

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Create the complete fraud detection pipeline
fraud_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]
)

# Train the model using the training dataset
fraud_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


Step 6 – Make predictions

In [6]:
# Predict fraud labels for test transactions
y_pred = fraud_model.predict(X_test)

# Predict fraud probabilities
y_pred_proba = fraud_model.predict_proba(X_test)[:, 1]

print("Predictions completed!")

Predictions completed!


Step 7 – Evaluate the model

In [ ]:
from sklearn.metrics import classification_report

# Compare predicted labels with actual labels
print(classification_report(
    y_test,
    y_pred,
    target_names=["Genuine", "Fraud"]
))

Print confusion matrix values

In [8]:
from sklearn.metrics import classification_report

# Compare predicted labels with actual labels
print(classification_report(
    y_test,
    y_pred,
    target_names=["Genuine", "Fraud"]
))

              precision    recall  f1-score   support

     Genuine       1.00      0.87      0.93   1108147
       Fraud       0.01      0.69      0.01      1493

    accuracy                           0.87   1109640
   macro avg       0.50      0.78      0.47   1109640
weighted avg       1.00      0.87      0.93   1109640



Save the trained model

In [9]:
import joblib

# Save the complete pipeline for use in the Streamlit app
model_path = PROJECT_DIR / "models" / "fraud_model.pkl"

joblib.dump(fraud_model, model_path)

print("Model saved successfully!")
print("Saved at:", model_path)

Model saved successfully!
Saved at: c:\kavya\redischool\fraud detection\models\fraud_model.pkl
